To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [blog post](https://unsloth.ai/blog/r1-reasoning) for guidance on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [2]:
%%capture
!apt-get install swi-prolog

In [3]:
!pip install -U pyswip

In [3]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip install unsloth vllm
!pip install --upgrade pillow

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [4]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Load up `Llama 3.1 8B Instruct`, and set parameters

In [5]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.6, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

INFO 02-26 16:24:32 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-3b-instruct-bnb-4bit with actual GPU utilization = 59.69%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.15 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 226.
Unsloth: vLLM's KV Cache can use up to 44.94 GB. Also swap space = 6 GB.
INFO 02-26 16:24:44 config.py:549] This model supports multiple tasks: {'generate', 'score', 'embed', 'reward', 'classif

[W226 16:24:46.624953978 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 02-26 16:24:46 loader.py:1089] Loading weights with BitsAndBytes quantization.  May take a while ...
INFO 02-26 16:24:47 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-26 16:24:51 model_runner.py:1115] Loading model weights took 1.9356 GB
INFO 02-26 16:24:51 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-26 16:24:58 worker.py:267] Memory profiling takes 4.49 seconds
INFO 02-26 16:24:58 worker.py:267] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.60) = 47.24GiB
INFO 02-26 16:24:58 worker.py:267] model weights take 1.94GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.24GiB; the rest of the memory reserved for KV Cache is 43.97GiB.
INFO 02-26 16:24:59 executor_base.py:111] # cuda blocks: 80051, # CPU blocks: 10922
INFO 02-26 16:24:59 executor_base.py:116] Maximum concurrency for 2048 tokens per request: 625.40x
INFO 02-26 16:25:10 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory erro

Capturing CUDA graph shapes: 100%|██████████| 32/32 [01:01<00:00,  1.91s/it]

INFO 02-26 16:26:11 model_runner.py:1562] Graph capturing finished in 61 secs, took 4.70 GiB
INFO 02-26 16:26:11 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 77.24 seconds



Unsloth 2025.2.15 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [6]:
import re
from datasets import load_dataset, Dataset
import ast
import concurrent
import os
from pyswip import Prolog
import multiprocessing

os.environ["WANDB_PROJECT"] = "prolog-3b"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# Load and prep dataset
SYSTEM_PROMPT = """
Generate a prolog solution for the asked question.
Follow these steps to craft your response:
1. reason about the given instruction
2. provide a high-quality prolog solution
3. write a query to verify the solution.
Output in the following format:
<reasoning>
...
</reasoning>
<code>
...
</code>
<query>
...
</query>

Write the query just inside <query></query> not in <code></code>.
"""

def extract_xml_knowledge(text: str) -> str:
    answer = text.split("<code>")[-1]
    answer = answer.split("</code>")[0]
    return answer.strip()

def extract_xml_query(text: str) -> str:
    answer = text.split("<query>")[-1]
    answer = answer.split("</query>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()


def split_prolog_rules(code):
    """
    Splits a string of multiple Prolog facts and rules by matching
    periods that are not part of a floating-point number. It removes
    any surrounding whitespace from each rule.
    """
    # This regex splits on a period that is NOT immediately followed by a digit.
    # \s* optionally matches any whitespace before and after the period.
    rules = re.split(r'\s*\.(?!\d)\s*', code)
    # Filter out any empty strings that may occur.
    return [rule for rule in rules if rule]

def parse_kb(prolog_code, query, answer):
    reward = 0
    try:
        prolog_interpreter = Prolog()
        # Filter out any empty rule
        #rules = [rule.strip() for rule in prolog_code.split(").") if rule.strip()]
        #rules = [rule + ")" for rule in rules]
        rules = split_prolog_rules(prolog_code)
        print("-"*20)
        print(rules)
        print("-"*20)
        for rule in rules:
            prolog_interpreter.assertz(rule)
            reward += 0.125
        # Use the instance's query method
        result = list(prolog_interpreter.query(query))
        for inference in result:
          for _, result_inference in inference.items():
            print("Infered: {}, Response: {}, Match: {}".format(result_inference, answer, float(result_inference) == float(answer)))
            try:
              if float(result_inference) == float(answer):
                return reward + 1
            except:
              print("Matching error!")
              return reward
        #print(result)
        # Ensure that the comparison makes sense:
        # This assumes you expect a non-empty result when the answer is correct.
        return reward + 0.25
    except Exception as e:
        print(f"Error encountered: {e}")
        return reward

def worker(prolog_code, query, answer):
  try:
    return parse_kb(prolog_code, query, answer)
  except:
    return 0

def eval_with_timeout(prolog_code, query, answer, env=None, timeout=5):
    if env is None:
        env = {}
    with concurrent.futures.ProcessPoolExecutor(max_workers=1) as executor:
        future = executor.submit(worker, prolog_code, query, answer)
        result = future.result(timeout=timeout)
        return 0 if result is None else result


def remove_prolog_comments_and_whitespace(code):
    # Remove block comments (/* ... */)
    code_no_block = re.sub(r'/\*[\s\S]*?\*/', '', code)
    # Remove single-line comments (% ...) from each line
    code_no_comments = re.sub(r'(?m)%.*$', '', code_no_block)
    # Remove newline and tab characters, but keep spaces
    cleaned_code = re.sub(r'[\n\t]', '', code_no_comments)
    return cleaned_code

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    reward = []
    for r, a in zip(responses, answer):
      knowledge_base = extract_xml_knowledge(r)
      knowledge_base = knowledge_base.replace("```prolog", "")
      knowledge_base = knowledge_base.replace("```", "")
      query = extract_xml_query(r)
      query = query.replace("```prolog", "")
      query = query.replace("```", "")
      query = query.replace("?-", "")
      query = query.strip()
      knowledge_base = remove_prolog_comments_and_whitespace(knowledge_base)
      print("#"*20)
      print(knowledge_base)
      print("#"*20)
      query = remove_prolog_comments_and_whitespace(query)
      reward_achieved = eval_with_timeout(knowledge_base, query, a)
      # check with ast if code can be parsed
      reward.append(reward_achieved)
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[-1]}", f"\nResponse:\n{responses[-1]}")
    return reward

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\s*([\s\S]*?)\s*</reasoning>\s*<code>\s*([\s\S]*?)\s*</code>\s*<query>\s*([\s\S]*?)\s*</query>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>\s*([\s\S]*?)\s*</reasoning>\s*<code>\s*([\s\S]*?)\s*</code>\s*<query>\s*([\s\S]*?)\s*</query>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("<code>\n") == 1:
        count += 0.125
    if text.count("\n</code>\n") == 1:
        count += 0.125
    if text.count("\n<query>\n") == 1:
        count += 0.125
    if text.count("\n</query>") == 1:
        count += 0.125
        count -= (len(text.split("\n</query>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [7]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 1024,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 1000,
    save_steps = 100,
    max_grad_norm = 0.1,
    report_to = "wandb", # Can use Weights & Biases
    output_dir="colab"
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [8]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        correctness_reward_func,
        strict_format_reward_func,
        soft_format_reward_func
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 4 | Gradient Accumulation steps = 1
\        /    Total batch size = 4 | Total steps = 1,000
 "-____-"     Number of trainable parameters = 59,867,136
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: federico-p98 (halykoss) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


####################
ticket_cost(40).total_cost(Tickets, DiscountedCost) :-    num_tickets(Tickets),    TotalTicketsWithoutDiscount is Tickets * ticket_cost,    ExcessTickets is max(0, Tickets - 10),    DiscountAmount is ExcessTickets * ticket_cost * 0.05,    DiscountedCost is TotalTicketsWithoutDiscount - DiscountAmount.total_cost(12, TotalCost).
####################
--------------------
['ticket_cost(40)', 'total_cost(Tickets, DiscountedCost) :-    num_tickets(Tickets),    TotalTicketsWithoutDiscount is Tickets * ticket_cost,    ExcessTickets is max(0, Tickets - 10),    DiscountAmount is ExcessTickets * ticket_cost * 0.05,    DiscountedCost is TotalTicketsWithoutDiscount - DiscountAmount', 'total_cost(12, TotalCost)']
--------------------
Error encountered: Caused by: 'total_cost(12, TotalCost).'. Returned: 'error(existence_error(procedure, /(num_tickets, 1)), context(/(total_cost, 2), _3432))'.
####################
ticket_price(40).tickets_bought(12).discount_percentage(5).discounte

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


--------------------
['ticket_price(40)', 'tickets_bought(12)', 'discount_percentage(5)', 'discounted_price(TicketPrice, DiscountedPrice) :-    ticket_price(TicketPrice),    DiscountedPrice is TicketPrice * (1 - DiscountPercentage / 100)', 'total_cost(TotalCost) :-    ticket_price(TicketPrice),    tickets_bought(TicketsBought),    DiscountedPrice is TicketPrice * (1 - DiscountPercentage / 100),    TotalCost is TicketsBought * DiscountedPrice', "query :-    total_cost(TotalCost),    write('Mr", "Benson paid a total of '), write(TotalCost), write(' dollars", "')"]
--------------------
Error encountered: Caused by: 'assertz((query :-    total_cost(TotalCost),    write('Mr)).'. Returned: 'error(syntax_error(end_of_file_in_quoted(')), string(b'a', 0))'.
####################
ticket_price(40).num_tickets(12).discounted_ticket_price(TicketPrice) :-  ticket_price(OriginalPrice),  num_tickets(TotalTickets),  ( TotalTickets > 10 ->    TicketPrice is OriginalPrice * 0.95  ; TicketPrice is Original

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
ERROR: Syntax error: End of file in quoted atom
ERROR: 
ERROR: ** here **
ERROR: a


...

ERROR: assertz/1: No permission to modify static procedure `(is)/2'


--------------------
['prob_winning_first := 0.30', 'prob_losing_first := 1 - prob_winning_first', 'prob_winning_second := 0.50', 'prob_losing_second := 1 - prob_winning_second', 'prob_losing_both := prob_losing_first * prob_losing_second', 'prob_winning_both := prob_winning_first * prob_winning_second', 'percentage_difference := (prob_losing_both - prob_winning_both) / prob_losing_both * 100', 'output(percentage_difference)']
--------------------


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-------------------- Question:
Andy is a lawyer who's working on two lawsuits. The first lawsuit has a 30% chance of paying out $5,000,000 upon a win and $0 if he loses it. The second lawsuit has a 50% chance of paying out $1,000,000 if Andy loses and a 50% chance of paying out $2,000,000 if he wins.  Expressed as a percentage, how much more likely is it that Andy loses both lawsuits compared to winning both of them? 
Answer:
20 
Response:
<reasoning>
To solve this problem, we need to calculate the probabilities of Andy losing both lawsuits and winning both lawsuits. Then, we will find the percentage difference between these probabilities.
Let's break down the steps:

1. **Calculate the probability of losing the first lawsuit:**
   - Probability of losing = 1 - Probability of winning = 1 - 0.30 = 0.70

2. **Calculate the probability of losing the second lawsuit:**
   - Probability of losing = 0.50

3. **Calculate the probability of losing both lawsuits:**
   - Probability of losing bot

KeyboardInterrupt: 

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
